In [ ]:
import pandas as pd
import numpy as np

# ---------------------------------------------------
# Step 0: Data Loading & Strict Pre-processing
# ---------------------------------------------------

# Load and combine both sheets from the Excel file
file_path = r"c:\Users\Administrator\Desktop\Coding\Unsupervised machine learning project\online_retail_II.xlsx"
all_sheets = pd.read_excel(file_path, sheet_name=None)
df = pd.concat(all_sheets.values(), ignore_index=True)

# Drop rows without Customer ID (we need a clear customer-level view)
df = df.dropna(subset=['Customer ID'])

# Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

# Convert Customer ID from float -> int -> string to avoid 12345.0 issue
df['Customer ID'] = df['Customer ID'].astype('int64').astype(str)

# Convert Invoice and StockCode to string
df['Invoice'] = df['Invoice'].astype(str)
df['StockCode'] = df['StockCode'].astype(str)

# Create is_canceled flag: True if invoice starts with 'C'
df['is_canceled'] = df['Invoice'].str.startswith('C')

# Masks for convenience
non_canceled_mask = ~df['is_canceled']
canceled_mask = df['is_canceled']

# Remove rows where Price <= 0 for non-canceled rows
df = df[~(non_canceled_mask & (df['Price'] <= 0))]

# For non-canceled orders, keep only positive quantities
df = df[~(non_canceled_mask & (df['Quantity'] <= 0))]

# For canceled orders, keep only negative quantities (returns/credits)
df = df[~(canceled_mask & (df['Quantity'] >= 0))]

# Remove non-product transactions
# - Strictly alphabetical stock codes (e.g., manual adjustments)
# - Known non-product codes (e.g., postage, bank charges)
non_product_codes = {
    'POST', 'D', 'M', 'CRUK', 'BANK CHARGES', 'DOT', 'CARRIAGE'
}

stockcode_upper = df['StockCode'].str.upper()
strict_alpha = stockcode_upper.str.isalpha()
is_manual_code = stockcode_upper.isin(non_product_codes)

df = df[~(strict_alpha | is_manual_code)].copy()

# Fill NaNs in categorical columns with 'Unknown' to avoid missing categories
categorical_cols = df.select_dtypes(include=['object', 'category']).columns
df[categorical_cols] = df[categorical_cols].fillna('Unknown')

# ---------------------------------------------------
# Step 1: Feature Engineering (Customer-Centric Dataset)
# ---------------------------------------------------

# LineTotal reflects revenue/credit at row level
df['LineTotal'] = df['Quantity'] * df['Price']

# Reference date for recency is the latest invoice date in the dataset
max_invoice_date = df['InvoiceDate'].max()

# ---------- Frequency ----------
# Number of unique successful invoices per customer
freq = (
    df.loc[~df['is_canceled']]
      .groupby('Customer ID')['Invoice']
      .nunique()
      .rename('Frequency')
)

# ---------- Monetary ----------
# Net spending including cancellations (negative LineTotal)
monetary = (
    df.groupby('Customer ID')['LineTotal']
      .sum()
      .rename('Monetary')
)

# ---------- Recency ----------
# Days since the last successful purchase
last_purchase_date = (
    df.loc[~df['is_canceled']]
      .groupby('Customer ID')['InvoiceDate']
      .max()
)

recency = (
    (max_invoice_date - last_purchase_date)
    .dt.days
    .rename('Recency')
)

# ---------- Variety ----------
# Unique products bought (successful orders only)
variety = (
    df.loc[~df['is_canceled']]
      .groupby('Customer ID')['StockCode']
      .nunique()
      .rename('Variety')
)

# ---------- Average Basket Size ----------
# Average number of items per successful invoice
basket_sizes = (
    df.loc[~df['is_canceled']]
      .groupby(['Customer ID', 'Invoice'])['Quantity']
      .sum()
)
avg_basket_size = (
    basket_sizes.groupby('Customer ID')
                .mean()
                .rename('AverageBasketSize')
)

# ---------- Cancellation Rate ----------
# Ratio of canceled invoices to all invoices per customer
total_invoices = df.groupby('Customer ID')['Invoice'].nunique()
canceled_invoices = (
    df.loc[df['is_canceled']]
      .groupby('Customer ID')['Invoice']
      .nunique()
)

cancellation_rate = (
    (canceled_invoices / total_invoices)
    .fillna(0.0)
    .rename('CancellationRate')
)

# ---------------------------------------------------
# Assemble customer-level dataset
# ---------------------------------------------------

customer_df = (
    pd.concat(
        [
            freq,
            monetary,
            recency,
            variety,
            avg_basket_size,
            cancellation_rate
        ],
        axis=1
    )
    .reset_index()
    .rename(columns={'Customer ID': 'CustomerID'})
)

# Enforce some intuitive dtypes
customer_df['CustomerID'] = customer_df['CustomerID'].astype(str)

# Frequency and Variety as nullable ints (customers with only cancellations might be NaN)
for col in ['Frequency', 'Variety']:
    if col in customer_df.columns:
        customer_df[col] = customer_df[col].astype('Int64')

# ---------------------------------------------------
# Outputs
# ---------------------------------------------------

print("customer_df.info():")
print(customer_df.info())
print("\ncustomer_df.head():")
print(customer_df.head())
print("\ncustomer_df.describe(include='all'):")
print(customer_df.describe(include='all'))